# 01 Generate Raw Tables

This notebook generates the **raw synthetic telemetry tables** for the Power BI Report Usage Forecasting project and saves them to `data/raw/`.

## Tables created

| Table | Description |
|---|---|
| `reports` | Report metadata — name, type, workspace, archetype |
| `users` | User reference data |
| `report_pages` | Page-level metadata per report |
| `dates` | Calendar dimension |
| `report_views` | Daily report-view events (one row per user × report × day) |
| `report_page_views` | Page-level events derived from report views |
| `report_load_times` | Load-time telemetry derived from report views |
| `report_archetypes` | Supplementary metadata: launch/retire dates, replacement links |

## Design philosophy

The data includes:

- Reports with unequal history lengths (launched mid-period or retired early)
- Irregular and sparse usage patterns
- Calendar-driven spikes (month-end, quarter-end)
- Usage cannibalisation after a replacement report launches
- Unpredictable executive-driven spikes

The `.py` script (`src/data/generate_synthetic_data.py`) is the operationalised version of this notebook — same logic, same outputs, no markdown. Read this notebook to understand *what* the data looks like and *why* it is designed this way; read the `.py` file to see the production implementation.


In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Add project root to path so we can import from src/
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

RAW_PATH = PROJECT_ROOT / "data" / "raw"
RAW_PATH.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Raw data path: {RAW_PATH}")


Project root : /Users/masegomodibane/Documents/GitHub/Data Science Projects /Forecasting Report Usage/GitHub Final Version/report-usage-forecasting
Raw data path: /Users/masegomodibane/Documents/GitHub/Data Science Projects /Forecasting Report Usage/GitHub Final Version/report-usage-forecasting/data/raw


## Simulation settings

These parameters control the scope of the dataset.
The random seed is passed to `numpy.random.default_rng` so every run is reproducible.


In [2]:
from src.data.generate_synthetic_data import (
    RANDOM_SEED, N_USERS, START_DATE, END_DATE, REPORT_CATALOG,
)

rng = np.random.default_rng(RANDOM_SEED)

all_dates = pd.date_range(START_DATE, END_DATE, freq="D")

print(f"Random seed : {RANDOM_SEED}")
print(f"Users       : {N_USERS}")
print(f"Date range  : {START_DATE} → {END_DATE}  ({len(all_dates)} days)")
print(f"Reports     : {len(REPORT_CATALOG)}  (3 per archetype × 10 archetypes)")


Random seed : 42
Users       : 200
Date range  : 2025-01-01 → 2026-03-31  (455 days)
Reports     : 30  (3 per archetype × 10 archetypes)


## Report catalogue and archetypes

Every report is defined individually in a catalogue with a realistic name, workspace, page list, and an **archetype** that controls how its usage time series is generated.

The 10 archetypes are:

| # | Archetype | What it models |
|---|---|---|
| 1 | `stable_weekly` | Consistent daily usage with weekday / weekend seasonality |
| 2 | `upward_adoption` | Growing user base over the period (logistic adoption curve) |
| 3 | `gradual_decline` | Slow erosion of usage as a better alternative emerges |
| 4 | `sudden_retirement` | Active until a hard cut-off date, then silent |
| 5 | `intermittent_specialist` | Sparse burst usage by a small specialist team |
| 6 | `month_end` | Quiet most of the month; heavy spike in the final 2–3 weekdays |
| 7 | `quarter_end` | Almost silent between quarters; intense spike at Mar / Jun / Sep / Dec |
| 8 | `launch_and_growth` | Zero history until mid-period launch date, then grows |
| 9 | `replacement_cannibalized` | Declining after a replacement report absorbs the user base |
| 10 | `high_volatility_exec` | Base readership plus unpredictable exec-meeting spikes |

The cell below loads the full catalogue and displays it as a table.


In [3]:
catalog_df = pd.DataFrame([
    {
        "report_id": e["report_id"],
        "report_name": e["report_name"],
        "archetype": e["archetype"],
        "workspace_id": e["workspace_id"],
        "report_type": e["report_type"],
        "n_pages": len(e["page_names"]),
    }
    for e in REPORT_CATALOG
])

catalog_df


,report_id,report_name,archetype,workspace_id,report_type,n_pages
0,R_001,Commercial Finance Exposure Dashboard,stable_weekly,WS_Finance,Dashboard,5
1,R_002,FX & Rates Risk Monitor,stable_weekly,WS_Risk,Report,6
2,R_003,Headcount & Payroll Summary,stable_weekly,WS_Finance,Report,5
3,R_004,New Business Volume Tracker,upward_adoption,WS_Commercial,Report,6
4,R_005,Digital Channel Adoption Report,upward_adoption,WS_Commercial,Report,5
5,R_006,ESG & Sustainability Metrics Dashboard,upward_adoption,WS_Risk,Dashboard,5
6,R_007,Legacy Cost Allocation Report,gradual_decline,WS_Finance,Report,4
7,R_008,Premises & Facilities Utilisation,gradual_decline,WS_Operations,Report,5
8,R_009,Manual Reconciliation Tracker,gradual_decline,WS_Finance,Paginated,3
9,R_010,Regulatory Capital Pre-Migration View,sudden_retirement,WS_Risk,Report,5


## Support tables

These tables provide reference data that stays constant across all archetypes.


### `users`

200 synthetic users with surrogate keys and email-style identifiers.
Activity weights are assigned later during event generation using a power-law distribution — a small number of users generate a disproportionate share of views, which is realistic for enterprise BI.


In [4]:
from src.data.generate_synthetic_data import generate_users

users = generate_users(N_USERS)
print(f"Shape: {users.shape}")
users.head()


Shape: (200, 3)


,user_key,user_id,unique_user
0,UK_0001,user001@masegoinc.com,User 001
1,UK_0002,user002@masegoinc.com,User 002
2,UK_0003,user003@masegoinc.com,User 003
3,UK_0004,user004@masegoinc.com,User 004
4,UK_0005,user005@masegoinc.com,User 005


### `dates`

A full calendar table covering the entire simulation window.
The `is_weekend` flag is used by every archetype generator to apply a weekday / weekend multiplier.


In [5]:
from src.data.generate_synthetic_data import generate_dates

dates = generate_dates(START_DATE, END_DATE)
print(f"Shape: {dates.shape}")
dates.head()


Shape: (455, 5)


,date,day_of_week,week_start_date,month,is_weekend
0,2025-01-01,Wednesday,2024-12-30,2025-01,False
1,2025-01-02,Thursday,2024-12-30,2025-01,False
2,2025-01-03,Friday,2024-12-30,2025-01,False
3,2025-01-04,Saturday,2024-12-30,2025-01,True
4,2025-01-05,Sunday,2024-12-30,2025-01,True


### `reports`

One row per report, populated directly from the catalogue.|


In [6]:
from src.data.generate_synthetic_data import generate_reports

reports = generate_reports(REPORT_CATALOG)
print(f"Shape: {reports.shape}")
reports


Shape: (30, 6)


,report_id,report_name,workspace_id,report_type,is_usage_metrics_report,archetype
0,R_001,Commercial Finance Exposure Dashboard,WS_Finance,Dashboard,False,stable_weekly
1,R_002,FX & Rates Risk Monitor,WS_Risk,Report,False,stable_weekly
2,R_003,Headcount & Payroll Summary,WS_Finance,Report,False,stable_weekly
3,R_004,New Business Volume Tracker,WS_Commercial,Report,False,upward_adoption
4,R_005,Digital Channel Adoption Report,WS_Commercial,Report,False,upward_adoption
5,R_006,ESG & Sustainability Metrics Dashboard,WS_Risk,Dashboard,False,upward_adoption
6,R_007,Legacy Cost Allocation Report,WS_Finance,Report,False,gradual_decline
7,R_008,Premises & Facilities Utilisation,WS_Operations,Report,False,gradual_decline
8,R_009,Manual Reconciliation Tracker,WS_Finance,Paginated,False,gradual_decline
9,R_010,Regulatory Capital Pre-Migration View,WS_Risk,Report,False,sudden_retirement


### `report_pages`

Page names come from the catalogue rather than generic `Page 1 / Page 2` labels.
Only `Report` and `Dashboard` type reports have pages (`Paginated` reports do not).


In [7]:
from src.data.generate_synthetic_data import generate_report_pages

report_pages = generate_report_pages(reports, REPORT_CATALOG)
print(f"Shape: {report_pages.shape}")
report_page_sample = report_pages[report_pages["report_id"] == "R_016"]
report_page_sample


Shape: (131, 3)


,report_id,section_id,section_name
55,R_016,R_016_P1,P&L Summary
56,R_016,R_016_P2,Revenue by Business Line
57,R_016,R_016_P3,Cost vs Budget
58,R_016,R_016_P4,Variance Commentary
59,R_016,R_016_P5,FX Impact
60,R_016,R_016_P6,YTD Performance
61,R_016,R_016_P7,Prior Month Comparison


## Archetype series generators

Each archetype is implemented as a function that takes the full date range and a `params` dict, and returns a **daily view-count array** (one float per day) for a single report.

The key design shift from the previous version:

| Old approach | New approach |
|---|---|
| For each date × report × user, draw a Bernoulli trial | For each report, generate a daily total array; then distribute that total across users |
| O(n_dates × n_reports × n_users) = 2.73 M iterations | O(n_reports × n_active_days × avg_users_per_day) ≈ 75 K iterations |
| All reports same shape | Each report has its own archetype-driven shape |

The helper functions below are shared across generators.


In [8]:
from src.data.generate_synthetic_data import (
    _weekend_mask,
    _days_to_month_end,
    _is_quarter_end_window,
    _apply_missing_telemetry_gaps,
    _ARCHETYPE_GENERATORS,
)

print("Available archetype generators:")
for name in _ARCHETYPE_GENERATORS:
    print(f"  {name}")


Available archetype generators:
  stable_weekly
  upward_adoption
  gradual_decline
  sudden_retirement
  intermittent_specialist
  month_end
  quarter_end
  launch_and_growth
  replacement_cannibalized
  high_volatility_exec


## Visualising each archetype

The cell below generates one representative daily series per archetype and prints a 60-day sample so you can eyeball the shape before committing to the full generation run.

Each bar represents one day; width is proportional to that day's view count.


In [9]:
# Pick a representative report from each archetype
showcase = {
    "stable_weekly":            ("R_001", "2025-06-02", "2025-06-29"),
    "upward_adoption":          ("R_004", "2025-01-01", "2025-03-01"),
    "gradual_decline":          ("R_007", "2025-01-01", "2025-03-01"),
    "sudden_retirement":        ("R_011", "2025-05-15", "2025-07-15"),
    "intermittent_specialist":  ("R_013", "2025-01-01", "2025-04-30"),
    "month_end":                ("R_016", "2025-02-22", "2025-03-07"),
    "quarter_end":              ("R_019", "2025-03-22", "2025-04-05"),
    "launch_and_growth":        ("R_024", "2025-07-01", "2025-07-31"),
    "replacement_cannibalized": ("R_025", "2025-06-25", "2025-09-10"),
    "high_volatility_exec":     ("R_028", "2025-04-01", "2025-04-30"),
}

catalog_map = {e["report_id"]: e for e in REPORT_CATALOG}
_rng_preview = np.random.default_rng(RANDOM_SEED)  # separate rng so it doesn't consume main seed

for archetype, (rid, start, end) in showcase.items():
    entry = catalog_map[rid]
    params = entry["params"]
    gen_fn = _ARCHETYPE_GENERATORS[archetype]

    series = gen_fn(all_dates, params, _rng_preview)
    series = np.clip(series, 0, None)

    mask = (all_dates >= pd.Timestamp(start)) & (all_dates <= pd.Timestamp(end))
    window_dates = all_dates[mask]
    window_vals  = series[mask]

    print(f"\n{'─' * 70}")
    print(f"  {archetype.upper()}  —  {entry['report_name']}")
    print(f"  Showing {start} → {end}")
    print(f"{'─' * 70}")
    for d, v in zip(window_dates, window_vals):
        bar = "█" * min(int(v / 4), 55)
        dow = d.day_name()[:3]
        print(f"  {d.date()}  {dow}  {v:>5.0f}  {bar}")



──────────────────────────────────────────────────────────────────────
  STABLE_WEEKLY  —  Commercial Finance Exposure Dashboard
  Showing 2025-06-02 → 2025-06-29
──────────────────────────────────────────────────────────────────────
  2025-06-02  Mon     33  ████████
  2025-06-03  Tue     41  ██████████
  2025-06-04  Wed     45  ███████████
  2025-06-05  Thu     62  ███████████████
  2025-06-06  Fri     46  ███████████
  2025-06-07  Sat     13  ███
  2025-06-08  Sun     10  ██
  2025-06-09  Mon     36  █████████
  2025-06-10  Tue     38  █████████
  2025-06-11  Wed     39  █████████
  2025-06-12  Thu     66  ████████████████
  2025-06-13  Fri     39  █████████
  2025-06-14  Sat     13  ███
  2025-06-15  Sun     10  ██
  2025-06-16  Mon     53  █████████████
  2025-06-17  Tue     48  ████████████
  2025-06-18  Wed     44  ██████████
  2025-06-19  Thu     45  ███████████
  2025-06-20  Fri     40  █████████
  2025-06-21  Sat     12  ███
  2025-06-22  Sun     10  ██
  2025-06-23  Mon    

## Generating all report-view events

`generate_all_views` iterates over the catalogue, calls the appropriate archetype generator for each report, then converts the daily totals into individual user-event rows.

**User assignment logic:**
For each `(date, report)` cell with views > 0, the function:
1. Draws the number of distinct active users from a Poisson distribution (scaled by the expected total views and an average views-per-user estimate).
2. Samples those users from the 200-user pool using **power-law weights** — so heavy users are more likely to appear, matching real enterprise usage patterns.
3. Splits the total view count across the selected users.

Specialist reports (`intermittent_specialist`) draw from a restricted pool of `n_specialist_users` to reflect that only a small team uses these reports.

This also generates `report_load_times` (one row per view event) as a by-product.


In [10]:
from src.data.generate_synthetic_data import generate_all_views

print("Generating report_views and report_load_times ...")
report_views, report_load_times = generate_all_views(REPORT_CATALOG, users, all_dates, rng)

print(f"report_views     : {report_views.shape}")
print(f"report_load_times: {report_load_times.shape}")
report_views.head()


Generating report_views and report_load_times ...


report_views     : (135430, 8)
report_load_times: (135430, 9)


,date,report_id,user_key,user_id,consumption_method,distribution_method,user_agent,view_count
0,2025-01-01,R_001,UK_0183,user183@masegoinc.com,Web,Direct,Chrome,12
1,2025-01-01,R_001,UK_0024,user024@masegoinc.com,Mobile,Direct,Edge,1
2,2025-01-01,R_001,UK_0180,user180@masegoinc.com,Mobile,SharedLink,Chrome,1
3,2025-01-01,R_001,UK_0013,user013@masegoinc.com,Web,App,Edge,1
4,2025-01-01,R_001,UK_0032,user032@masegoinc.com,Web,Direct,Edge,1


## Generating report-page-view events

Page-view events are derived from the report-view table.
For each `(report, date, user)` view event, we randomly select 1–N pages from that report's page list (using the actual named pages from the catalogue, not generic `Page 1 / Page 2` labels).


In [11]:
from src.data.generate_synthetic_data import generate_report_page_views

print("Generating report_page_views ...")
report_page_views = generate_report_page_views(report_views, report_pages, rng)

print(f"report_page_views: {report_page_views.shape}")
report_page_views.head()


Generating report_page_views ...


report_page_views: (270635, 8)


,timestamp,date,report_id,section_id,user_key,client,session_source,page_view_count
0,2025-01-01 07:47:00,2025-01-01,R_001,R_001_P3,UK_0183,Browser,App,1
1,2025-01-01 01:04:00,2025-01-01,R_001,R_001_P1,UK_0183,Browser,Direct,1
2,2025-01-01 14:16:00,2025-01-01,R_001,R_001_P2,UK_0183,Browser,App,1
3,2025-01-01 12:00:00,2025-01-01,R_001,R_001_P4,UK_0183,Browser,Direct,1
4,2025-01-01 02:43:00,2025-01-01,R_001,R_001_P5,UK_0183,Browser,App,1


## Archetype metadata table

`report_archetypes` is a supplementary table (not present in the previous version).
It records each report's launch date, retire date, and which report (if any) it replaces.

This table enables model-performance analysis broken out by archetype:
*"How does SARIMA perform on `month_end` reports vs `intermittent_specialist` reports?"*


In [12]:
from src.data.generate_synthetic_data import generate_report_archetypes

report_archetypes = generate_report_archetypes(REPORT_CATALOG, all_dates)
report_archetypes


,report_id,report_name,archetype,launch_date,retire_date,replaces_report_id,history_start,history_end
0,R_001,Commercial Finance Exposure Dashboard,stable_weekly,2025-01-01,NaN,NaN,2025-01-01,2026-03-31
1,R_002,FX & Rates Risk Monitor,stable_weekly,2025-01-01,NaN,NaN,2025-01-01,2026-03-31
2,R_003,Headcount & Payroll Summary,stable_weekly,2025-01-01,NaN,NaN,2025-01-01,2026-03-31
3,R_004,New Business Volume Tracker,upward_adoption,2025-01-01,NaN,NaN,2025-01-01,2026-03-31
4,R_005,Digital Channel Adoption Report,upward_adoption,2025-01-01,NaN,NaN,2025-01-01,2026-03-31
5,R_006,ESG & Sustainability Metrics Dashboard,upward_adoption,2025-01-01,NaN,NaN,2025-01-01,2026-03-31
6,R_007,Legacy Cost Allocation Report,gradual_decline,2025-01-01,NaN,NaN,2025-01-01,2026-03-31
7,R_008,Premises & Facilities Utilisation,gradual_decline,2025-01-01,NaN,NaN,2025-01-01,2026-03-31
8,R_009,Manual Reconciliation Tracker,gradual_decline,2025-01-01,NaN,NaN,2025-01-01,2026-03-31
9,R_010,Regulatory Capital Pre-Migration View,sudden_retirement,2025-01-01,2025-09-12,NaN,2025-01-01,2025-09-12


## Validation

Basic referential integrity checks to confirm the generated tables are coherent before saving.


In [13]:
checks = {
    "All report_ids in report_views exist in reports":
        report_views["report_id"].isin(reports["report_id"]).all(),

    "All user_keys in report_views exist in users":
        report_views["user_key"].isin(users["user_key"]).all(),

    "All section_ids in report_page_views exist in report_pages":
        report_page_views["section_id"].isin(report_pages["section_id"]).all(),

    "All report_ids in report_load_times exist in reports":
        report_load_times["report_id"].isin(reports["report_id"]).all(),

    "No null dates in report_views":
        report_views["date"].isnull().sum() == 0,

    "No null view_counts in report_views":
        report_views["view_count"].isnull().sum() == 0,
}

all_passed = True
for msg, result in checks.items():
    status = "✓ PASS" if result else "✗ FAIL"
    print(f"  [{status}]  {msg}")
    if not result:
        all_passed = False

print()
print("All checks passed." if all_passed else "One or more checks FAILED.")


  [✓ PASS]  All report_ids in report_views exist in reports
  [✓ PASS]  All user_keys in report_views exist in users
  [✓ PASS]  All section_ids in report_page_views exist in report_pages
  [✓ PASS]  All report_ids in report_load_times exist in reports
  [✓ PASS]  No null dates in report_views
  [✓ PASS]  No null view_counts in report_views

All checks passed.


## Summary by archetype

This table shows the key statistics per report, grouped by archetype.
It confirms that the archetypes produce meaningfully different data shapes.

Key things to look for:
- **`active_days`** — `intermittent_specialist` should be very low; `stable_weekly` should be high
- **`history_days`** — `launch_and_growth` and `sudden_retirement` should be shorter than 455
- **`max_daily`** — `quarter_end` and `month_end` should have extreme spikes; `stable_weekly` should not
- **`median_daily`** — `quarter_end` should have a very low median (near-silence between spikes)


In [14]:
daily = (
    report_views
    .groupby(["report_id", "date"])["view_count"]
    .sum()
    .reset_index()
    .rename(columns={"view_count": "daily_views"})
)

stats = daily.groupby("report_id").agg(
    first_active=("date", "min"),
    last_active=("date", "max"),
    active_days=("daily_views", lambda x: (x > 0).sum()),
    total_views=("daily_views", "sum"),
    median_daily=("daily_views", "median"),
    max_daily=("daily_views", "max"),
).reset_index()

stats["history_days"] = (
    pd.to_datetime(stats["last_active"]) - pd.to_datetime(stats["first_active"])
).dt.days + 1

summary = (
    stats
    .merge(report_archetypes[["report_id", "report_name", "archetype"]], on="report_id")
    [["report_id", "report_name", "archetype", "history_days", "active_days",
      "total_views", "median_daily", "max_daily"]]
    .sort_values(["archetype", "report_id"])
    .reset_index(drop=True)
)

pd.set_option("display.max_colwidth", 45)
pd.set_option("display.float_format", "{:.1f}".format)
summary


,report_id,report_name,archetype,history_days,active_days,total_views,median_daily,max_daily
0,R_007,Legacy Cost Allocation Report,gradual_decline,455,438,10546,21.0,76
1,R_008,Premises & Facilities Utilisation,gradual_decline,454,435,6651,15.0,55
2,R_009,Manual Reconciliation Tracker,gradual_decline,455,435,4415,9.0,36
3,R_028,Group CEO Weekly Intelligence Briefing,high_volatility_exec,455,449,9452,13.0,369
4,R_029,Board Risk Committee Dashboard,high_volatility_exec,455,446,5470,8.0,192
5,R_030,Investment Committee Decision Support,high_volatility_exec,455,445,4584,7.0,178
6,R_013,Model Validation Audit Log,intermittent_specialist,434,49,575,11.0,24
7,R_014,Stress Testing Scenario Browser,intermittent_specialist,437,34,647,20.0,36
8,R_015,Regulatory Submission Evidence Pack,intermittent_specialist,427,37,395,12.0,18
9,R_022,AI & Automation Benefits Tracker,launch_and_growth,352,338,11875,35.0,102


In [15]:
# Archetype-level summary (collapsed across 3 reports per archetype)
archetype_summary = (
    summary
    .groupby("archetype")
    .agg(
        avg_history_days=("history_days", "mean"),
        avg_active_days=("active_days", "mean"),
        avg_total_views=("total_views", "mean"),
        avg_median_daily=("median_daily", "mean"),
        avg_max_daily=("max_daily", "mean"),
    )
    .round(1)
    .reset_index()
    .sort_values("archetype")
)

archetype_summary


,archetype,avg_history_days,avg_active_days,avg_total_views,avg_median_daily,avg_max_daily
0,gradual_decline,454.7,436.0,7204.0,15.0,55.7
1,high_volatility_exec,455.0,446.7,6502.0,9.3,246.3
2,intermittent_specialist,432.7,40.0,539.0,14.3,26.0
3,launch_and_growth,307.7,296.3,11751.7,39.0,119.7
4,month_end,455.0,325.0,4857.7,7.7,169.3
5,quarter_end,455.0,325.0,2652.7,2.7,221.0
6,replacement_cannibalized,454.7,425.3,9805.7,16.0,101.7
7,stable_weekly,455.0,426.0,12285.0,32.3,61.3
8,sudden_retirement,255.0,255.0,4885.7,21.7,39.7
9,upward_adoption,455.0,428.7,7926.3,15.3,71.3


## Save raw tables

All tables are saved to `data/raw/` as CSV files.



In [16]:
tables = {
    "reports":            reports,
    "users":              users,
    "report_pages":       report_pages,
    "dates":              dates,
    "report_views":       report_views,
    "report_page_views":  report_page_views,
    "report_load_times":  report_load_times,
    "report_archetypes":  report_archetypes,
}

for name, df in tables.items():
    path = RAW_PATH / f"{name}.csv"
    df.to_csv(path, index=False)
    print(f"  Saved {path.name:30s}  {df.shape}")


  Saved reports.csv                     (30, 6)
  Saved users.csv                       (200, 3)
  Saved report_pages.csv                (131, 3)
  Saved dates.csv                       (455, 5)


  Saved report_views.csv                (135430, 8)


  Saved report_page_views.csv           (270635, 8)


  Saved report_load_times.csv           (135430, 9)
  Saved report_archetypes.csv           (30, 8)


## Next step

With the raw tables saved, the next notebook builds the **clean semantic model**:

`02_build_semantic_model_csv.ipynb`

It transforms the raw event tables into:

- `dim_date`, `dim_user`, `dim_report`, `dim_page`  — dimension tables
- `fact_report_views`, `fact_page_views`, `fact_report_loads`  — fact tables

Those processed tables feed into feature engineering, forecasting, and the Streamlit app.


